# Task 3 — understand the training data

The assignment's first check (section 5.7.1) asks us to print the first few rows of **MMLU and SmolTalk**, then answer:

1. **How many training examples does each dataset contribute?**
2. **What does a single training example look like?**

We also inspect **GSM-8K**, because it is the other Stage 1 dataset.

| Stage | Datasets | What we want the model to practise |
|---|---|---|
| 1: mid-training | MMLU + GSM-8K | Multiple-choice knowledge questions and worked maths solutions |
| 2: supervised fine-tuning (SFT) | SmolTalk | Following instructions and holding conversations |

This notebook uses nanochat's existing dataset classes. It inspects data and saves evidence; it does not train a model. Later we will train both stages from our depth-2 model and compare their benchmark scores.


## 1. Set up the notebook

In VS Code, select the Python interpreter **`task2/.venv/bin/python`** as the notebook kernel. We reuse that environment; CUDA is not needed here.

The next cell finds the project, makes nanochat importable, and keeps downloads in `task3/cache`. It supports our current folder layout and a checkout with the task folders inside nanochat.


In [3]:
import os
import sys
import json
import csv
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import Markdown, display

# Find the project even when the notebook starts inside task3/.
ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'task3').is_dir()
    and ((parent / 'nanochat' / 'tasks').is_dir() or (parent / 'tasks').is_dir())
)
NANOCHAT = ROOT / 'nanochat' if (ROOT / 'nanochat' / 'tasks').is_dir() else ROOT
TASK3 = ROOT / 'task3'
CACHE = TASK3 / 'cache'
RESULTS = TASK3 / 'results'
CACHE.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

# Import the existing loaders without editing or copying their code.
sys.dont_write_bytecode = True
sys.path.insert(0, str(NANOCHAT))
os.environ['NANOCHAT_BASE_DIR'] = str(CACHE)
print('Python:', sys.executable)
print('Results folder:', RESULTS)


Python: /home/s4784863/agentic_assignment1/task2/.venv/bin/python
Results folder: /home/s4784863/agentic_assignment1/task3/results


## 2. Load the training splits

These are the exact dataset choices in `scripts/chat_sft.py`:

- **MMLU:** `cais/mmlu`, configuration `all`, split `auxiliary_train`.
- **GSM-8K:** `openai/gsm8k`, configuration `main`, split `train`.
- **SmolTalk:** `HuggingFaceTB/smol-smoltalk`, configuration `default`, split `train`.

Nanochat uses the smaller **smol-smoltalk** dataset. Its size should not be confused with the full SmolTalk collection. We load complete training splits, so our counts are measured from the actual data. The first run downloads Parquet files; later runs reuse them.

Each loader shuffles rows using **seed 42**. Below, “first rows” means the first rows in that reproducible shuffled order. We do not load evaluation splits here.


In [2]:
from tasks.mmlu import MMLU
from tasks.gsm8k import GSM8K
from tasks.smoltalk import SmolTalk

mmlu = MMLU(subset='all', split='auxiliary_train')
gsm8k = GSM8K(subset='main', split='train')
smoltalk = SmolTalk(split='train')

tasks = {'MMLU': mmlu, 'GSM-8K': gsm8k, 'SmolTalk': smoltalk}
for name, task in tasks.items():
    print(f'{name}: {len(task):,} training rows; columns: {task.ds.table.column_names}')


MMLU: 99,842 training rows; columns: ['answer', 'choices', 'question', 'subject']
GSM-8K: 7,473 training rows; columns: ['question', 'answer']
SmolTalk: 460,341 training rows; columns: ['messages', 'source']


## 3. Inspect rows and training conversations

There are two views of the same example:

- **`task.ds[index]`:** the original dataset fields, before conversation formatting.
- **`task[index]`:** the user/assistant conversation that nanochat passes to its tokenizer.

`print_rows()` below prints a few original rows without shortening their contents. This is our only display helper; the dataset classes do the conversion for us.

**Experiment:** change `EXAMPLE_INDEX` to inspect a different training conversation. Change `N_PREVIEW` to print more or fewer original rows. Rerun the cells below after changing them.


In [4]:
N_PREVIEW = 3
EXAMPLE_INDEX = 0


def print_rows(task, count=3):
    """Print a few raw dataset rows so we can inspect their fields and contents."""
    for index in range(min(count, len(task))):
        print(f'Row {index}:')
        print(json.dumps(task.ds[index], indent=2, ensure_ascii=False))
        print()


assert N_PREVIEW >= 1
assert 0 <= EXAMPLE_INDEX < min(len(task) for task in tasks.values())


### MMLU: a question and four choices

A raw row has a `question`, four `choices`, an `answer` index, and a `subject` field. Answer indices **0, 1, 2, 3** correspond to **A, B, C, D**.

Nanochat puts the question and choices into the user message. The assistant's target is **one answer letter**, not an explanation. Notice that nanochat places each letter *after* its choice, such as `choice text=A`.


In [5]:
print_rows(mmlu, N_PREVIEW)


Row 0:
{
  "answer": 1,
  "choices": [
    "Ten hours.",
    "Nine hours.",
    "Seven hours.",
    "Eight hours."
  ],
  "question": "Rules in the reading room Hello, everyone. Welcome to the school reading room. We hope you have a good time here. Before you go into the reading room, there are some rules you need to keep. 1.The reading room is open from 8:00 a.m. to 5:00 p.m. from Monday to Friday. 2. Don't take your bag into the reading room. 3. Don't talk loudly in the reading room. 4. Don't take any food or drink into the reading room. 5. Take only one book at a time. After you finish reading the book, you must put it back and then you can take another one. Don't take many books to your seat. 6. Before you leave, you must the book to the bookshelf. You can't take any book out of the reading room. How long is the reading room open every day?",
  "subject": ""
}

Row 1:
{
  "answer": 1,
  "choices": [
    "We can place an order securely with the help of search engine.",
    "The offi

In [6]:
# The formatted training conversation for our chosen row.
print(json.dumps(mmlu[EXAMPLE_INDEX], indent=2, ensure_ascii=False))


{
  "messages": [
    {
      "role": "user",
      "content": "Multiple Choice question: Rules in the reading room Hello, everyone. Welcome to the school reading room. We hope you have a good time here. Before you go into the reading room, there are some rules you need to keep. 1.The reading room is open from 8:00 a.m. to 5:00 p.m. from Monday to Friday. 2. Don't take your bag into the reading room. 3. Don't talk loudly in the reading room. 4. Don't take any food or drink into the reading room. 5. Take only one book at a time. After you finish reading the book, you must put it back and then you can take another one. Don't take many books to your seat. 6. Before you leave, you must the book to the bookshelf. You can't take any book out of the reading room. How long is the reading room open every day?\n- Ten hours.=A\n- Nine hours.=B\n- Seven hours.=C\n- Eight hours.=D\n\nRespond only with the letter of the correct answer."
    },
    {
      "role": "assistant",
      "content": "B"
  

### GSM-8K: a maths question and worked answer

The raw `answer` contains solution steps and a final answer after `####`.

Nanochat converts expressions between `<<` and `>>` into `python` and `python_output` parts. The assistant's content is therefore a **list of parts**, rather than a plain string. Nothing is executed by this inspection: we are looking at the stored expression and result.


In [7]:
print_rows(gsm8k, N_PREVIEW)


Row 0:
{
  "question": "Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?",
  "answer": "Mimi has 2 x 12 = <<2*12=24>>24 sea shells.\nKyle has 24 x 2 = <<24*2=48>>48 sea shells.\nLeigh has 48 / 3 = <<48/3=16>>16 sea shells.\n#### 16"
}

Row 1:
{
  "question": "Frankie's parents let him have many pets. He has six more snakes than he has cats. He has one less parrot than cats. Six of his pets have four legs. He has 2 dogs. How many pets does he have in total?",
  "answer": "He has 6 - 2 = <<6-2=4>>4 cats.\nHe has 4 - 1 = <<4-1=3>>3 parrots.\nHe has 4 + 6 = <<4+6=10>>10 snakes.\nHe has a total of 2 + 4 + 3 + 10 = <<2+4+3+10=19>>19 pets.\n#### 19"
}

Row 2:
{
  "question": "Olaf collects colorful toy cars. At first, his collection consisted of 150 cars. His family, knowing his hobby, decided to give him some toy cars. Grandpa gave Olaf

In [8]:
print(json.dumps(gsm8k[EXAMPLE_INDEX], indent=2, ensure_ascii=False))


{
  "messages": [
    {
      "role": "user",
      "content": "Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?"
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "text",
          "text": "Mimi has 2 x 12 = "
        },
        {
          "type": "python",
          "text": "2*12"
        },
        {
          "type": "python_output",
          "text": "24"
        },
        {
          "type": "text",
          "text": "24 sea shells.\nKyle has 24 x 2 = "
        },
        {
          "type": "python",
          "text": "24*2"
        },
        {
          "type": "python_output",
          "text": "48"
        },
        {
          "type": "text",
          "text": "48 sea shells.\nLeigh has 48 / 3 = "
        },
        {
          "type": "python",
          "text": "48/3"
        }

### SmolTalk: a conversation

A raw row already contains `messages`, with a `role` and `content` for each message. Nanochat retains those messages and checks their structure. A conversation can start with a system message, followed by alternating user and assistant messages.

**One dataset row is one conversation.** A row can contain several assistant replies, so counting rows is different from counting messages or tokens.


In [9]:
print_rows(smoltalk, N_PREVIEW)


Row 0:
{
  "messages": [
    {
      "content": "A researcher interested in examining students' study time per day collected data from a random sample of 20 students. Here are the data: \n\n3, 1, 2, 4, 5, 2, 3, 4, 2, 1, 2, 3, 4, 6, 3, 2, 2, 1, 5, 4\n\nWhat is the mean of this dataset? How would you describe or interpret the result?",
      "role": "user"
    },
    {
      "content": "To calculate the mean, we need to add up all the values and divide by the total number of observations. \n\nThe sum of the values is 3 + 1 + 2 + 4 + 5 + 2 + 3 + 4 + 2 + 1 + 2 + 3 + 4 + 6 + 3 + 2 + 2 + 1 + 5 + 4 = 59.\n\nThere are 20 observations in the dataset. The mean is calculated by dividing the sum by the number of observations: 59 / 20 = 2.95.\n\nThe mean of 2.95 hours per day indicates that, on average, students in this sample spend approximately 3 hours per day studying. This result suggests a relatively moderate amount of daily study time, with most students likely dedicating a few hours to study

In [10]:
print(json.dumps(smoltalk[EXAMPLE_INDEX], indent=2, ensure_ascii=False))


{
  "messages": [
    {
      "content": "A researcher interested in examining students' study time per day collected data from a random sample of 20 students. Here are the data: \n\n3, 1, 2, 4, 5, 2, 3, 4, 2, 1, 2, 3, 4, 6, 3, 2, 2, 1, 5, 4\n\nWhat is the mean of this dataset? How would you describe or interpret the result?",
      "role": "user"
    },
    {
      "content": "To calculate the mean, we need to add up all the values and divide by the total number of observations. \n\nThe sum of the values is 3 + 1 + 2 + 4 + 5 + 2 + 3 + 4 + 2 + 1 + 2 + 3 + 4 + 6 + 3 + 2 + 2 + 1 + 5 + 4 = 59.\n\nThere are 20 observations in the dataset. The mean is calculated by dividing the sum by the number of observations: 59 / 20 = 2.95.\n\nThe mean of 2.95 hours per day indicates that, on average, students in this sample spend approximately 3 hours per day studying. This result suggests a relatively moderate amount of daily study time, with most students likely dedicating a few hours to studying eac

## 4. How many examples does each dataset contribute?

First count **rows before repetition**. These are dataset records; we have not checked whether their text contains duplicates.

Then calculate the contribution if we retain nanochat's current default mixture repetitions: **MMLU × 3**, **GSM-8K × 4**, and **SmolTalk × 1**. For the assignment, MMLU and GSM-8K belong to Stage 1; SmolTalk belongs to Stage 2. The original trainer currently mixes all three together, which we will adapt separately later.

**Contribution = training rows × repetitions.** Repetition reuses existing examples; it does not create new questions. These are planned mixture entries for a full pass, not examples already processed by a trained model, optimizer steps, or a token budget.

**Experiment:** change the repetition values to `1` and rerun this cell. How does the balance of Stage 1 change? These settings affect this counting experiment only.


In [11]:
MMLU_REPETITIONS = 3
GSM8K_REPETITIONS = 4
SMOLTALK_REPETITIONS = 1
assert all(isinstance(n, int) and n >= 1 for n in (
    MMLU_REPETITIONS, GSM8K_REPETITIONS, SMOLTALK_REPETITIONS
))

settings = [
    ('MMLU', 'Stage 1', 'cais/mmlu', 'all', 'auxiliary_train', MMLU_REPETITIONS),
    ('GSM-8K', 'Stage 1', 'openai/gsm8k', 'main', 'train', GSM8K_REPETITIONS),
    ('SmolTalk', 'Stage 2', 'HuggingFaceTB/smol-smoltalk', 'default', 'train', SMOLTALK_REPETITIONS),
]
counts = []
for name, stage, repo, config, split, repetitions in settings:
    rows = len(tasks[name])
    counts.append(dict(dataset=name, stage=stage, repo=repo, config=config,
                       split=split, training_rows=rows, repetitions=repetitions,
                       mixture_entries=rows * repetitions))

table = '| Dataset | Stage | Training rows | Repetitions | Mixture entries |\n'
table += '|---|---|---:|---:|---:|\n'
for row in counts:
    table += f"| {row['dataset']} | {row['stage']} | {row['training_rows']:,} | {row['repetitions']} | {row['mixture_entries']:,} |\n"
display(Markdown(table))
for stage in ('Stage 1', 'Stage 2'):
    total = sum(row['mixture_entries'] for row in counts if row['stage'] == stage)
    print(f'{stage}: {total:,} mixture entries for one complete pass')


| Dataset | Stage | Training rows | Repetitions | Mixture entries |
|---|---|---:|---:|---:|
| MMLU | Stage 1 | 99,842 | 3 | 299,526 |
| GSM-8K | Stage 1 | 7,473 | 4 | 29,892 |
| SmolTalk | Stage 2 | 460,341 | 1 | 460,341 |


Stage 1: 329,418 mixture entries for one complete pass
Stage 2: 460,341 mixture entries for one complete pass


## 5. Compare what one example contains

The next cell counts messages and assistant replies in the **selected example only**. It is not a dataset-wide average.

MMLU asks the model to predict a letter. GSM-8K supplies worked solutions. SmolTalk can supply several conversational replies within one row. Therefore, equal numbers of rows do not necessarily provide equal numbers of target tokens.


In [12]:
for name, task in tasks.items():
    messages = task[EXAMPLE_INDEX]['messages']
    assistant_replies = sum(message['role'] == 'assistant' for message in messages)
    roles = ' -> '.join(message['role'] for message in messages)
    print(f'{name}, example {EXAMPLE_INDEX}: {len(messages)} messages, {assistant_replies} assistant replies')
    print(f'  {roles}')


MMLU, example 0: 2 messages, 1 assistant replies
  user -> assistant
GSM-8K, example 0: 2 messages, 1 assistant replies
  user -> assistant
SmolTalk, example 0: 6 messages, 3 assistant replies
  user -> assistant -> user -> assistant -> user -> assistant


## 6. Save our inspection results

This cell saves the counts, raw previews, and selected formatted examples. It also records the nanochat source version and checksums of the downloaded files, so we can identify the data we actually inspected.

Nanochat's loader fetches the current Hub Parquet export on the first download and reuses its cache. It does **not** pin a dataset revision. The recorded SHA-256 checksums identify our downloaded snapshot; they are not a promise that a fresh download months later will contain identical bytes.

Rerunning this cell replaces the saved inspection results with the current notebook settings. Save the notebook too (`Ctrl+S`) to keep its displayed outputs.


In [13]:
import subprocess
import importlib.metadata


def file_sha256(path):
    """Identify a downloaded file by hashing its bytes in small chunks."""
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


examples = {}
shards = []
for row in counts:
    task = tasks[row['dataset']]
    examples[row['dataset']] = {
        'preview_raw_rows': [task.ds[i] for i in range(min(N_PREVIEW, len(task)))],
        'example_index': EXAMPLE_INDEX,
        'selected_raw_row': task.ds[EXAMPLE_INDEX],
        'selected_conversation': task[EXAMPLE_INDEX],
    }
    shard_dir = CACHE / 'task_data' / row['repo'].replace('/', '--') / row['config'] / row['split']
    for filename in json.loads((shard_dir / 'manifest.json').read_text()):
        path = shard_dir / filename
        shards.append(dict(dataset=row['dataset'], file=str(path.relative_to(TASK3)),
                           bytes=path.stat().st_size, sha256=file_sha256(path)))

commit = subprocess.run(['git', '-C', str(NANOCHAT), 'rev-parse', 'HEAD'],
                        capture_output=True, text=True)
record = dict(
    inspected_at_utc=datetime.now(timezone.utc).isoformat(),
    nanochat_commit=commit.stdout.strip() if commit.returncode == 0 else None,
    shuffle_seed=42, preview_rows=N_PREVIEW, example_index=EXAMPLE_INDEX,
    dataset_revision_pinned=False,
    packages={name: importlib.metadata.version(name) for name in ('numpy', 'pyarrow', 'torch')},
    counts=counts, shards=shards,
)
with (RESULTS / 'dataset_counts.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(counts[0]))
    writer.writeheader()
    writer.writerows(counts)
for filename, value in [('examples.json', examples), ('inspection.json', record)]:
    (RESULTS / filename).write_text(json.dumps(value, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Saved dataset_counts.csv, examples.json, and inspection.json in', RESULTS)


Saved dataset_counts.csv, examples.json, and inspection.json in /home/s4784863/agentic_assignment1/task3/results


## 7. Your observations

Use the printed evidence to answer in your own words:

1. How many training rows does each dataset contain? How does repetition change its contribution?
2. What is the user input and the assistant target in each dataset?
3. Which dataset provides answer letters? Which provides worked maths solutions? Which provides conversations?
4. Why might one SmolTalk row give more assistant target tokens than one MMLU row?
5. What languages, topics, styles, or assumptions do you notice in the examples? What would you need to inspect before making claims about the whole dataset?

**My notes:**

_Write your observations here after trying a few example indices._

### What comes later in Task 3

This notebook covers the initial data inspection. The assignment also requires two training stages and saved checkpoints; a concrete token-level explanation of assistant-only loss masking; ARC-Easy, ARC-Challenge, and GSM-8K scores for the base, mid-trained, and SFT models; dataset analysis; and a conceptual comparison with LoRA/QLoRA.

Dataset sources: [MMLU](https://huggingface.co/datasets/cais/mmlu), [GSM-8K](https://huggingface.co/datasets/openai/gsm8k), [smol-smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk). The exact loader calls and transformations are in nanochat's `tasks/mmlu.py`, `tasks/gsm8k.py`, `tasks/smoltalk.py`, and `tasks/common.py`.
